# Attention Is All You Need — Walkthrough

**Paper:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017)

This notebook walks through the Transformer architecture, connecting each paper section to code.
All examples use **tiny dimensions** so everything runs instantly on CPU.
The architecture is identical to the paper — only the numbers change.

For each component:
1. Paper excerpt (the relevant passage)
2. Code (the implementation)
3. Sanity check (verify shapes and behavior)

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# Toy dimensions — paper uses d_model=512, we use 64 for CPU walkthrough
BATCH_SIZE = 2
SRC_LEN = 10
TGT_LEN = 8
D_MODEL = 64      # Paper: 512
N_HEADS = 4        # Paper: 8
D_FF = 128         # Paper: 2048
N_LAYERS = 2       # Paper: 6
VOCAB_SIZE = 100   # Paper: 37000
DROPOUT = 0.0      # Set to 0 for deterministic walkthrough

print(f"Using toy dimensions: d_model={D_MODEL}, n_heads={N_HEADS}, d_ff={D_FF}")
print(f"d_k = d_model / n_heads = {D_MODEL // N_HEADS}")

## 1. Scaled Dot-Product Attention (§3.2.1)

> "An attention function can be described as mapping a query and a set of key-value
> pairs to an output, where the query, keys, values, and output are all vectors.
> The output is computed as a weighted sum of the values, where the weight assigned
> to each value is computed by a compatibility function of the query with the
> corresponding key."
>
> — §3.2, Attention Is All You Need

**Equation 1:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """§3.2.1, Eq. 1 — Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V"""
    d_k = query.size(-1)
    
    # (batch, n_heads, seq_q, d_k) @ (batch, n_heads, d_k, seq_k) -> (batch, n_heads, seq_q, seq_k)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)  # (batch, n_heads, seq_q, seq_k)
    return torch.matmul(attn_weights, value)   # (batch, n_heads, seq_q, d_v)

In [ ]:
# Sanity check: scaled dot-product attention
d_k = D_MODEL // N_HEADS
q = torch.randn(BATCH_SIZE, N_HEADS, SRC_LEN, d_k)
k = torch.randn(BATCH_SIZE, N_HEADS, SRC_LEN, d_k)
v = torch.randn(BATCH_SIZE, N_HEADS, SRC_LEN, d_k)

attn_out = scaled_dot_product_attention(q, k, v)
assert attn_out.shape == (BATCH_SIZE, N_HEADS, SRC_LEN, d_k), \
    f"Expected {(BATCH_SIZE, N_HEADS, SRC_LEN, d_k)}, got {attn_out.shape}"
print(f"✓ Scaled dot-product attention: ({BATCH_SIZE}, {N_HEADS}, {SRC_LEN}, {d_k}) -> {attn_out.shape}")

# Verify attention weights sum to 1
scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
weights = F.softmax(scores, dim=-1)
weight_sums = weights.sum(dim=-1)
assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-6), \
    "Attention weights should sum to 1"
print(f"✓ Attention weights sum to 1 (verified)")

## 2. Multi-Head Attention (§3.2.2)

> "Multi-head attention allows the model to jointly attend to information from
> different representation subspaces at different positions. With a single attention
> head, averaging inhibits this."
>
> "MultiHead(Q, K, V) = Concat(head₁, ..., headₕ)W^O
>  where headᵢ = Attention(QWᵢ^Q, KWᵢ^K, VWᵢ^V)"
>
> "We employ h = 8 parallel attention layers, or heads. For each of these we use
> d_k = d_v = d_model/h = 64."
>
> — §3.2.2, Attention Is All You Need

In [ ]:
class MultiHeadAttention(nn.Module):
    """§3.2.2 — Multi-Head Attention."""
    
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # §3.2.2 — d_k = d_model/h
        
        # §3.2.2 — projection matrices W_Q, W_K, W_V, W_O
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Project and reshape: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
        q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        attn_out = scaled_dot_product_attention(q, k, v, mask=mask)
        
        # Concatenate and project: (batch, n_heads, seq, d_k) -> (batch, seq, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_out)

In [ ]:
# Sanity check: multi-head attention
mha = MultiHeadAttention(D_MODEL, N_HEADS, dropout=0.0)
x = torch.randn(BATCH_SIZE, SRC_LEN, D_MODEL)
out = mha(x, x, x)  # self-attention
assert out.shape == (BATCH_SIZE, SRC_LEN, D_MODEL), \
    f"Expected {(BATCH_SIZE, SRC_LEN, D_MODEL)}, got {out.shape}"
print(f"✓ MultiHeadAttention: (batch={BATCH_SIZE}, seq={SRC_LEN}, d_model={D_MODEL}) -> {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in mha.parameters()):,}")
print(f"  Expected: 4 * d_model^2 + 4 * d_model (biases) = {4 * D_MODEL**2 + 4 * D_MODEL:,}")

## 3. Position-wise Feed-Forward Network (§3.3)

> "In addition to attention sub-layers, each of the layers in our encoder and decoder
> contains a fully connected feed-forward network, which is applied to each position
> separately and identically."
>
> "FFN(x) = max(0, xW₁ + b₁)W₂ + b₂" (Eq. 2)
>
> "The dimensionality of input and output is d_model = 512, and the inner-layer has
> dimensionality d_ff = 2048."
>
> — §3.3, Attention Is All You Need

In [ ]:
class FeedForward(nn.Module):
    """§3.3, Eq. 2 — FFN(x) = max(0, xW_1 + b_1)W_2 + b_2"""
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # §3.3, Eq. 2 — ReLU activation
        x = self.linear1(x)     # (batch, seq, d_model) -> (batch, seq, d_ff)
        x = F.relu(x)           # max(0, ...)
        x = self.dropout(x)
        x = self.linear2(x)     # (batch, seq, d_ff) -> (batch, seq, d_model)
        return x

# Sanity check
ff = FeedForward(D_MODEL, D_FF)
x = torch.randn(BATCH_SIZE, SRC_LEN, D_MODEL)
out = ff(x)
assert out.shape == (BATCH_SIZE, SRC_LEN, D_MODEL)
print(f"✓ FeedForward: {x.shape} -> {out.shape}")
print(f"  Inner dimension: {D_FF} (expansion factor: {D_FF/D_MODEL}x)")

## 4. Positional Encoding (§3.5)

> "Since our model contains no recurrence and no convolution, in order for the model
> to make use of the order of the sequence, we must inject some information about the
> relative or absolute position of the tokens in the sequence."
>
> "We use sine and cosine functions of different frequencies:
>   PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
>   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))"
>
> — §3.5, Attention Is All You Need

In [ ]:
class PositionalEncoding(nn.Module):
    """§3.5 — Sinusoidal positional encoding."""
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # odd: cos
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Sanity check
pos_enc = PositionalEncoding(D_MODEL, max_len=100)
x = torch.randn(BATCH_SIZE, SRC_LEN, D_MODEL)
out = pos_enc(x)
assert out.shape == x.shape
print(f"✓ PositionalEncoding: {x.shape} -> {out.shape} (added, not concatenated)")

# Verify PE values are bounded (sin/cos are in [-1, 1])
pe_values = pos_enc.pe[0, :SRC_LEN, :]
assert pe_values.min() >= -1.0 and pe_values.max() <= 1.0
print(f"✓ PE values bounded in [-1, 1]: min={pe_values.min():.3f}, max={pe_values.max():.3f}")

## 5. Encoder Layer (§3.1)

> "The encoder is composed of a stack of N = 6 identical layers. Each layer has two
> sub-layers. The first is a multi-head self-attention mechanism, and the second is a
> simple, position-wise fully connected feed-forward network."
>
> "We employ a residual connection around each of the two sub-layers, followed by
> layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x))."
>
> — §3.1, Attention Is All You Need

In [ ]:
class EncoderLayer(nn.Module):
    """§3.1 — Single encoder layer: self-attention + feed-forward, with residual + norm."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, norm_eps=1e-6):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        # [UNSPECIFIED] LayerNorm epsilon not stated in paper — using 1e-6
        self.norm1 = nn.LayerNorm(d_model, eps=norm_eps)
        self.norm2 = nn.LayerNorm(d_model, eps=norm_eps)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # §3.1 — Post-norm: LayerNorm(x + Sublayer(x))
        attn_out = self.self_attn(x, x, x, mask)  # self-attention
        x = self.norm1(x + self.dropout1(attn_out))
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_out))
        return x

# Sanity check
enc_layer = EncoderLayer(D_MODEL, N_HEADS, D_FF)
x = torch.randn(BATCH_SIZE, SRC_LEN, D_MODEL)
out = enc_layer(x)
assert out.shape == (BATCH_SIZE, SRC_LEN, D_MODEL)
print(f"✓ EncoderLayer: {x.shape} -> {out.shape}")

## 6. Decoder Layer (§3.1)

> "In addition to the two sub-layers in each encoder layer, the decoder inserts a
> third sub-layer, which performs multi-head attention over the output of the encoder
> stack."
>
> "We also modify the self-attention sub-layer in the decoder stack to prevent
> positions from attending to subsequent positions. This masking, combined with the
> fact that the output embeddings are offset by one position, ensures that the
> predictions for position i can depend only on the known outputs at positions less
> than i."
>
> — §3.1, §3.2.3, Attention Is All You Need

In [ ]:
class DecoderLayer(nn.Module):
    """§3.1 — Single decoder layer: masked self-attn + cross-attn + FFN."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0, norm_eps=1e-6):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=norm_eps)
        self.norm2 = nn.LayerNorm(d_model, eps=norm_eps)
        self.norm3 = nn.LayerNorm(d_model, eps=norm_eps)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, memory, tgt_mask=None, memory_mask=None):
        # Sub-layer 1: Masked self-attention
        self_attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(self_attn_out))
        # Sub-layer 2: Cross-attention over encoder output
        cross_attn_out = self.cross_attn(x, memory, memory, memory_mask)
        x = self.norm2(x + self.dropout2(cross_attn_out))
        # Sub-layer 3: Feed-forward
        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_out))
        return x

# Sanity check
dec_layer = DecoderLayer(D_MODEL, N_HEADS, D_FF)
x = torch.randn(BATCH_SIZE, TGT_LEN, D_MODEL)
memory = torch.randn(BATCH_SIZE, SRC_LEN, D_MODEL)

# Causal mask for decoder self-attention
causal_mask = torch.tril(torch.ones(TGT_LEN, TGT_LEN, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
out = dec_layer(x, memory, tgt_mask=causal_mask)
assert out.shape == (BATCH_SIZE, TGT_LEN, D_MODEL)
print(f"✓ DecoderLayer: tgt {x.shape} + memory {memory.shape} -> {out.shape}")

## 7. Full Transformer (§3)

> "Most competitive neural sequence transduction models have an encoder-decoder
> structure. Here, the encoder maps an input sequence of symbol representations
> (x₁, ..., xₙ) to a sequence of continuous representations z = (z₁, ..., zₙ).
> Given z, the decoder then generates an output sequence (y₁, ..., yₘ) of symbols
> one element at a time."
>
> — §2, Attention Is All You Need

> "In the embedding layers, we multiply those weights by √d_model."
> "In our model, we share the same weight matrix between the two embedding layers
> and the pre-softmax linear transformation."
>
> — §3.4, Attention Is All You Need

In [ ]:
class Transformer(nn.Module):
    """§3 — Full Transformer encoder-decoder."""
    
    def __init__(self, d_model, n_heads, d_ff, n_layers, vocab_size,
                 max_len=5000, dropout=0.0, norm_eps=1e-6, tie_weights=True):
        super().__init__()
        
        # §3.4 — Embeddings
        self.src_embed = nn.Embedding(vocab_size, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        if tie_weights:  # §3.4 — "share the same weight matrix"
            self.tgt_embed.weight = self.src_embed.weight
        
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.embed_scale = math.sqrt(d_model)  # §3.4 — multiply by sqrt(d_model)
        
        # §3.1 — Encoder: N identical layers
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout, norm_eps)
            for _ in range(n_layers)
        ])
        
        # §3.1 — Decoder: N identical layers
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout, norm_eps)
            for _ in range(n_layers)
        ])
        
        # §3.4 — Output projection (tied with embeddings)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.output_proj.weight = self.src_embed.weight
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        tgt_len = tgt.size(1)
        
        # Auto-generate causal mask if not provided
        if tgt_mask is None:
            tgt_mask = torch.tril(torch.ones(tgt_len, tgt_len, 
                         dtype=torch.bool, device=tgt.device)).unsqueeze(0).unsqueeze(0)
        
        # §3.4 — Embed and scale by sqrt(d_model)
        src_emb = self.pos_encoding(self.src_embed(src) * self.embed_scale)
        tgt_emb = self.pos_encoding(self.tgt_embed(tgt) * self.embed_scale)
        
        # Encode
        memory = src_emb
        for layer in self.encoder_layers:
            memory = layer(memory, src_mask)
        
        # Decode
        output = tgt_emb
        for layer in self.decoder_layers:
            output = layer(output, memory, tgt_mask, src_mask)
        
        # §3.4 — Project to vocabulary
        logits = self.output_proj(output)  # (batch, tgt_len, vocab_size)
        return logits

In [ ]:
# Sanity check: full model forward pass
model = Transformer(
    d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
    n_layers=N_LAYERS, vocab_size=VOCAB_SIZE,
    dropout=0.0,
)

src = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SRC_LEN))
tgt = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, TGT_LEN))
logits = model(src, tgt)

assert logits.shape == (BATCH_SIZE, TGT_LEN, VOCAB_SIZE), \
    f"Expected {(BATCH_SIZE, TGT_LEN, VOCAB_SIZE)}, got {logits.shape}"
print(f"✓ Full Transformer: src {src.shape} + tgt {tgt.shape} -> logits {logits.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {total_params:,}")
print(f"  (Paper base model has ~65M params with d_model=512, vocab=37k)")

## 8. Label-Smoothed Loss (§5.4)

> "During training, we employed label smoothing of value ε_ls = 0.1. This hurt
> perplexity, as the model learns to be more unsure, but improved accuracy and
> BLEU score."
>
> — §5.4, Attention Is All You Need

In [ ]:
def label_smoothed_cross_entropy(logits, targets, smoothing=0.1, pad_idx=0):
    """§5.4 — Label smoothing with ε_ls = 0.1."""
    vocab_size = logits.size(-1)
    logits = logits.contiguous().view(-1, vocab_size)
    targets = targets.contiguous().view(-1)
    
    log_probs = F.log_softmax(logits, dim=-1)
    nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    smooth_loss = -log_probs.mean(dim=-1)
    
    loss = (1.0 - smoothing) * nll_loss + smoothing * smooth_loss
    non_pad = targets != pad_idx
    return (loss * non_pad.float()).sum() / non_pad.float().sum().clamp(min=1.0)

# Sanity check: loss computes and backprops
targets = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, TGT_LEN))  # no padding (idx 0)
loss = label_smoothed_cross_entropy(logits, targets, smoothing=0.1)
loss.backward()
print(f"✓ Label-smoothed loss: {loss.item():.4f}")
print(f"  (Compare: -log(1/V) = {-math.log(1/VOCAB_SIZE):.4f} for random predictions)")

## 9. Gradient Check

Verify that gradients flow through all parameters of the model.

In [ ]:
# Check gradients exist for all parameters
no_grad_params = []
for name, param in model.named_parameters():
    if param.grad is None:
        no_grad_params.append(name)

if no_grad_params:
    print(f"⚠ Parameters without gradients: {no_grad_params}")
else:
    print(f"✓ All {sum(1 for _ in model.parameters())} parameter tensors have gradients")
    
# Check gradient norms
total_norm = 0.0
for param in model.parameters():
    if param.grad is not None:
        total_norm += param.grad.data.norm(2).item() ** 2
total_norm = total_norm ** 0.5
print(f"  Total gradient norm: {total_norm:.4f}")

## Common Pitfalls

1. **Pre-norm vs post-norm**: The paper's Figure 1 shows post-norm (`LayerNorm(x + Sublayer(x))`), and §3.1 says "followed by layer normalization" which also implies post-norm. However, many modern reimplementations use pre-norm (`x + Sublayer(LayerNorm(x))`) because it's more stable for deep models. If you're having trouble training, try switching to pre-norm. [§3.1]

2. **Attention scale factor**: Must divide by √d_k where d_k = d_model/n_heads, NOT √d_model. For the base model: √64 = 8, not √512 ≈ 22.6. This is a 3x difference that significantly affects attention sharpness. [§3.2.1, Eq. 1]

3. **Positional encoding frequencies**: The div_term uses `exp(arange(0, d_model, 2) * (-log(10000) / d_model))`, stepping by 2. Using `arange(0, d_model)` (step 1) doubles the frequency count and gives wrong wavelengths. [§3.5]

4. **Label smoothing**: ε_ls = 0.1 means the true class gets probability 1 - ε + ε/V ≈ 0.9 (not exactly 0.9). PyTorch's `CrossEntropyLoss(label_smoothing=0.1)` uses a slightly different formula — for large V this is negligible but worth knowing. [§5.4]

5. **Learning rate schedule**: The Transformer schedule (Eq. 3) computes the LR from scratch — it's not a multiplier on a base LR. Set the optimizer LR to 1.0 and let the schedule compute the actual LR. Peaks at ~step 4000 for d_model=512. [§5.3, Eq. 3]

6. **Embedding scaling**: The paper multiplies embedding weights by √d_model (§3.4). This is easy to forget and affects the relative magnitude of embeddings vs positional encodings.

7. **Weight tying**: The paper shares weights between (a) source embedding, (b) target embedding, and (c) output projection. Missing this changes the parameter count and can affect quality. [§3.4]